# 🧠 Mini-R1: Reproducing DeepSeek R1's "Aha Moment" with GRPO

**Train a language model to develop step-by-step reasoning using only reward signals.**

In this notebook, we reproduce one of the most exciting findings from the DeepSeek R1 paper: during reinforcement learning training, models spontaneously learn to "think step by step" — the so-called **"aha moment"**. We use **Group Relative Policy Optimization (GRPO)** to train `Qwen/Qwen2.5-3B-Instruct` on arithmetic puzzles, and watch reasoning emerge.

> **Prerequisites**: Basic familiarity with PyTorch, Transformers, and the concept of RL fine-tuning (RLHF). Understanding of what a reward function does.

**Objectives:**
- ✅ Understand the GRPO algorithm and how it differs from PPO
- ✅ Implement rule-based reward functions for format and mathematical correctness
- ✅ Train a model to develop emergent reasoning on a single GPU (Colab) or multi-GPU (RunPod)
- ✅ Observe the "aha moment" — the model starts self-correcting and thinking step by step
- ✅ Scale training to 8 GPUs using DeepSpeed ZeRO-3 + vLLM


## 📑 Table of Contents

1. [Background: The Aha Moment & GRPO](#1)
2. [Environment Setup](#2)
3. [The Countdown Game — Our Training Task](#3)
4. [Dataset Loading & Prompt Engineering](#4)
5. [Reward Function Design](#5)
6. [Testing Our Reward Functions](#6)
7. [Single-GPU Training with GRPOTrainer (Colab)](#7)
8. [Multi-GPU Training Script & Configs](#8)
9. [Inspecting Results](#9)
10. [Visualizing Training Progress](#10)
11. [Key Takeaways](#11)
12. [Further Reading](#12)


<a name="1"></a>
## 1. 💡 Background: The Aha Moment & GRPO

### What is the "Aha Moment"?

DeepSeek R1 discovered something remarkable: when you train a language model with RL (using only simple reward signals), the model **spontaneously learns to reason step by step**. Without any chain-of-thought demonstrations, the model starts:

1. Breaking problems into sub-steps inside `<think>` tags
2. Trying different approaches and backtracking
3. Self-verifying its answers before committing

This emergent behavior was dubbed the **"aha moment"** — the model discovers thinking on its own.

### What is GRPO?

**Group Relative Policy Optimization** is a simpler alternative to PPO:

| | PPO | GRPO |
|---|---|---|
| Critic model | Required (same size as policy) | **Not needed** |
| Baseline | Value function $V(s)$ | **Group mean reward** |
| Memory | 2x model memory | **1x model memory** |
| Advantage | $A = R - V(s)$ | $\hat{A}_i = \frac{r_i - \text{mean}(r)}{\text{std}(r)}$ |

**How GRPO works:**
1. For each prompt, generate $G$ outputs (a "group")
2. Score each output with reward functions
3. Normalize rewards within the group (subtract mean, divide by std)
4. Outputs better than group average → increase probability
5. Outputs worse than group average → decrease probability

$$\hat{A}_i = \frac{r_i - \text{mean}(\{r_1, ..., r_G\})}{\text{std}(\{r_1, ..., r_G\})}$$


<a name="2"></a>
## 2. ⚙️ Environment Setup

> **Run this cell first** to install all required libraries. This takes 3-5 minutes on Colab.


In [ ]:
# Install PyTorch (Colab usually has this, but pinning version for reproducibility)
!pip install -q "torch==2.5.1" tensorboard "setuptools<71.0.0" --index-url https://download.pytorch.org/whl/cu121

# Install Flash Attention for efficient attention computation
!pip install -q flash-attn --no-build-isolation

# Install Hugging Face ecosystem + RL training libraries
!pip install -q --upgrade \
    "transformers==4.48.1" \
    "datasets==3.1.0" \
    "accelerate==1.3.0" \
    "hf-transfer==0.1.9" \
    "deepspeed==0.15.4" \
    "trl==0.14.0"

# Install vLLM for fast generation during GRPO
!pip install -q "vllm==0.7.0"

# For single-GPU QLoRA training on Colab
!pip install -q "peft==0.14.0" "bitsandbytes==0.45.0"


In [ ]:
# Verify installations
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

import transformers, datasets, accelerate, trl
print(f"Transformers: {transformers.__version__}")
print(f"TRL: {trl.__version__}")
print(f"Accelerate: {accelerate.__version__}")


<a name="3"></a>
## 3. 🎲 The Countdown Game — Our Training Task

The **Countdown Game** is an arithmetic puzzle:

> Given a set of numbers (e.g., `[19, 36, 55, 7]`) and a target (e.g., `65`), create an equation using `+`, `-`, `*`, `/` that equals the target. Each number can be used **exactly once**.

**Example:**
```
Numbers: [19, 36, 55, 7]  Target: 65
Solution: 55 + 36 - 19 - 7 = 65  ✓
```

This is perfect for RL training because:
- Solutions are **verifiable** (just evaluate the equation)
- Multiple valid solutions may exist
- Requires **reasoning** to find the right combination
- Simple enough for a 3B model, hard enough to be interesting


In [ ]:
# Let's look at what a Countdown puzzle looks like
from datasets import load_dataset

# Load the dataset
raw_dataset = load_dataset("Jiayi-Pan/Countdown-Tasks-3to4", split="train")
print(f"Total puzzles: {len(raw_dataset):,}")
print(f"\nFeatures: {raw_dataset.features}")

# Show a few examples
for i in range(5):
    ex = raw_dataset[i]
    print(f"\nPuzzle {i+1}: Numbers={ex['nums']}, Target={ex['target']}")


<a name="4"></a>
## 4. 📦 Dataset Loading & Prompt Engineering

We need to format each puzzle as a conversation prompt that:
1. Tells the model the task
2. Asks it to show work in `<think>` tags
3. **Pre-fills** the assistant response with `<think>` to nudge the model into reasoning mode

This "prefix" trick is key — we start the assistant's response with `"Let me solve this step by step.\n<think>"` so the model continues from there.


In [ ]:
from transformers import AutoTokenizer
from datasets import load_dataset

# Load dataset
DATASET_ID = "Jiayi-Pan/Countdown-Tasks-3to4"
dataset = load_dataset(DATASET_ID, split="train")
dataset = dataset.shuffle(seed=42).select(range(50_000))  # 50k samples
print(f"Selected {len(dataset):,} samples")

# Load tokenizer
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
print(f"Tokenizer loaded: {MODEL_ID}")


In [ ]:
def generate_r1_prompt(numbers, target):
    """Create a reasoning prompt with <think> prefix for the model."""
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant. You first thinks about the "
                "reasoning process in the mind and then provides the user "
                "with the answer."
            ),
        },
        {
            "role": "user",
            "content": (
                f"Using the numbers {numbers}, create an equation that "
                f"equals {target}. You can use basic arithmetic operations "
                f"(+, -, *, /) one or multiple times but each number can "
                f"only be used once. Show your work in <think> </think> "
                f"tags. And return the final equation in <answer> </answer> "
                f"tags, for example <answer> (1 + 2) / 3 </answer>. "
                f"Think step by step inside <think> tags."
            ),
        },
        {
            "role": "assistant",
            "content": "Let me solve this step by step.\n<think>",
        },
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, continue_final_message=True
    )
    return {"prompt": prompt, "target": target, "nums": numbers}


# Apply to dataset
dataset = dataset.map(lambda x: generate_r1_prompt(x["nums"], x["target"]))

# Train/test split
split = dataset.train_test_split(test_size=0.1)
train_dataset = split["train"]
test_dataset = split["test"]

print(f"Train: {len(train_dataset):,} | Test: {len(test_dataset):,}")


In [ ]:
# Let's look at a formatted prompt
sample = train_dataset[0]
print("=" * 60)
print("FORMATTED PROMPT:")
print("=" * 60)
print(sample["prompt"])
print("=" * 60)
print(f"\nTarget: {sample['target']}")
print(f"Available numbers: {sample['nums']}")


<a name="5"></a>
## 5. 🎯 Reward Function Design

We use **two rule-based reward functions** — no neural reward model needed:

### Reward 1: Format Check
Does the output follow `<think>...</think>\n<answer>...</answer>` format?
- **1.0** if format is correct
- **0.0** if format is wrong

### Reward 2: Equation Correctness
Is the equation mathematically correct?
1. Extract equation from `<answer>` tags
2. Check: all given numbers used exactly once
3. Check: only valid operators (+, -, *, /, parentheses)
4. Evaluate: does result equal target?
- **1.0** if all checks pass
- **0.0** if any check fails

> 💡 **Key insight**: These simple binary rewards are sufficient to drive the entire learning process. The model learns format first (~50 steps), then accuracy (~100+ steps), then genuine reasoning (~300+ steps).


In [ ]:
import re
import os
import random


def format_reward_func(completions, target, **kwargs):
    """
    Check if output follows <think>...</think>\n<answer>...</answer> format.
    Returns 1.0 for correct format, 0.0 otherwise.
    """
    rewards = []
    for completion, gt in zip(completions, target):
        try:
            # Add synthetic <think> (already in prompt prefix)
            completion = "<think>" + completion

            # Log 10% of samples for inspection
            if random.random() < 0.1:
                os.makedirs("completion_samples", exist_ok=True)
                with open("completion_samples/completion_samples.txt", "a") as f:
                    f.write(f"\n\n==============\n{completion}")

            # Regex: <think>...</think>\n<answer>...</answer>
            regex = (
                r"^<think>([^<]*(?:<(?!/?think>)[^<]*)*)"
                r"<\/think>\n<answer>([\s\S]*?)<\/answer>$"
            )
            match = re.search(regex, completion, re.DOTALL)
            if match is None or len(match.groups()) != 2:
                rewards.append(0.0)
            else:
                rewards.append(1.0)
        except Exception:
            rewards.append(0.0)
    return rewards


In [ ]:
def equation_reward_func(completions, target, nums, **kwargs):
    """
    Check if the equation in <answer> tags is mathematically correct.
    Validates: correct format, all numbers used once, result matches target.
    Returns 1.0 for correct equation, 0.0 otherwise.
    """
    rewards = []
    for completion, gt, numbers in zip(completions, target, nums):
        try:
            completion = "<think>" + completion

            # Step 1: Extract equation from <answer> tags
            match = re.search(r"<answer>(.*?)<\/answer>", completion)
            if match is None:
                rewards.append(0.0)
                continue
            equation = match.group(1).strip()

            # Step 2: Check all numbers used exactly once
            used_numbers = [int(n) for n in re.findall(r'\d+', equation)]
            if sorted(used_numbers) != sorted(numbers):
                rewards.append(0.0)
                continue

            # Step 3: Only allow numbers, operators, parens, whitespace
            allowed_pattern = r'^[\d+\-*/().\s]+$'
            if not re.match(allowed_pattern, equation):
                rewards.append(0.0)
                continue

            # Step 4: Evaluate and check result
            result = eval(equation, {"__builtins__": None}, {})
            if abs(float(result) - float(gt)) < 1e-5:
                rewards.append(1.0)
                # Log successful samples
                if random.random() < 0.10:
                    os.makedirs("completion_samples", exist_ok=True)
                    with open("completion_samples/success_samples.txt", "a") as f:
                        f.write(f"\n\n==============\n{completion}")
            else:
                rewards.append(0.0)
        except Exception:
            rewards.append(0.0)
    return rewards


<a name="6"></a>
## 6. 🧪 Testing Our Reward Functions

Before training, let's verify our reward functions work correctly with known examples.


In [ ]:
# Test cases
correct = (
    "I need to find 65 from [19, 36, 55, 7]. "
    "Let me try: 55 + 36 - 19 - 7 = 65. Yes! </think>\n"
    "<answer> 55 + 36 - 19 - 7 </answer>"
)

wrong_format = (
    "The answer is 55 + 36 - 19 - 7 = 65. "
    "I found it by trying different combinations."
)

wrong_math = (
    "Let me add them up: 55 + 36 + 19 - 7 = 103 </think>\n"
    "<answer> 55 + 36 + 19 - 7 </answer>"
)

wrong_numbers = (
    "I'll use 55 and 10: 55 + 10 = 65 </think>\n"
    "<answer> 55 + 10 </answer>"
)

test_completions = [correct, wrong_format, wrong_math, wrong_numbers]
test_targets = ["65"] * 4
test_nums = [[19, 36, 55, 7]] * 4
test_labels = ["Correct", "Wrong format", "Wrong math", "Wrong numbers"]


In [ ]:
# Run reward functions on test cases
format_rewards = format_reward_func(test_completions, test_targets)
equation_rewards = equation_reward_func(test_completions, test_targets, test_nums)

print(f"{'Sample':<20} {'Format':>8} {'Equation':>10} {'Total':>8}")
print("-" * 50)
for label, fr, er in zip(test_labels, format_rewards, equation_rewards):
    total = fr + er
    status = '✅' if total == 2.0 else '⚠️' if total == 1.0 else '❌'
    print(f"{label:<20} {fr:>8.1f} {er:>10.1f} {total:>8.1f}  {status}")

# Verify expected results
assert format_rewards == [1.0, 0.0, 1.0, 1.0], f"Format rewards wrong: {format_rewards}"
assert equation_rewards == [1.0, 0.0, 0.0, 0.0], f"Equation rewards wrong: {equation_rewards}"
print("\n✅ All reward function tests passed!")


<a name="7"></a>
## 7. 🚀 Single-GPU Training with GRPOTrainer (Colab)

For a quick demo on Colab's single GPU, we use:
- **QLoRA** (4-bit quantization + LoRA adapters) to fit in 16-40GB
- **Reduced settings**: `num_generations=2`, `max_steps=100`
- **No vLLM** (single GPU can't spare one for generation)

> ⚠️ This will be slow (~5-20 min/step on Colab T4/A100). For real training, use the multi-GPU setup in Section 8.
> You can reduce `max_steps` to 10-20 just to verify the pipeline works.


In [ ]:
from trl import GRPOConfig, GRPOTrainer, get_peft_config, ModelConfig

# Model config with QLoRA for memory efficiency
model_config = ModelConfig(
    model_name_or_path="Qwen/Qwen2.5-3B-Instruct",
    torch_dtype="bfloat16",
    attn_implementation="flash_attention_2",
    use_peft=True,       # Enable LoRA
    load_in_4bit=True,   # 4-bit quantization
)

# Training hyperparameters (reduced for single GPU)
training_args = GRPOConfig(
    output_dir="qwen-r1-aha-moment",
    learning_rate=5e-7,
    lr_scheduler_type="cosine",
    logging_steps=1,
    max_steps=20,                # Increase to 100+ for real training
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    bf16=True,
    # GRPO specific
    max_prompt_length=256,
    max_completion_length=512,   # Shorter for speed
    num_generations=2,           # Small group (8 for full training)
    beta=0.001,                  # Low KL penalty
    # No vLLM on single GPU
    use_vllm=False,
)

print("Config created!")
print(f"  Model: {model_config.model_name_or_path}")
print(f"  Steps: {training_args.max_steps}")
print(f"  Generations per prompt: {training_args.num_generations}")
print(f"  Max completion length: {training_args.max_completion_length}")


In [ ]:
# Create the GRPO trainer
trainer = GRPOTrainer(
    model=model_config.model_name_or_path,
    reward_funcs=[format_reward_func, equation_reward_func],
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    peft_config=get_peft_config(model_config),
)

print("✅ GRPOTrainer initialized!")
print(f"  Trainable params: {trainer.model.print_trainable_parameters()}")


In [ ]:
# Train! (This will be slow on single GPU)
print("Starting training...")
train_result = trainer.train()

# Print results
print("\n" + "=" * 50)
print("Training complete!")
for key, value in train_result.metrics.items():
    print(f"  {key}: {value}")

# Save model
trainer.save_model(training_args.output_dir)
print(f"\nModel saved to: {training_args.output_dir}")


<a name="8"></a>
## 8. 🖥️ Multi-GPU Training Script & Configs

For **real training** on 4-8 GPUs (e.g., RunPod, Lambda, etc.), we use:
- **DeepSpeed ZeRO Stage 3** — shards optimizer, gradients, AND parameters
- **vLLM** on the last GPU for fast generation
- **HuggingFace Accelerate** to orchestrate everything

### Architecture (8 GPUs)
```
GPU 0-6: DeepSpeed ZeRO-3 (training)
GPU 7:   vLLM (generation)

Scaling rule: For N GPUs, set num_processes = N-1
```

The cells below write all the needed files to disk.


In [ ]:
# Write the DeepSpeed + Accelerate config
import os
os.makedirs("configs", exist_ok=True)

deepspeed_config = """compute_environment: LOCAL_MACHINE
debug: false
deepspeed_config:
  deepspeed_multinode_launcher: standard
  offload_optimizer_device: none
  offload_param_device: none
  zero3_init_flag: true
  zero3_save_16bit_model: true
  zero_stage: 3
distributed_type: DEEPSPEED
downcast_bf16: 'no'
machine_rank: 0
main_training_function: main
mixed_precision: bf16
num_machines: 1
num_processes: 7  # N_GPUs - 1 (last GPU reserved for vLLM)
rdzv_backend: static
same_network: true
tpu_env: []
tpu_use_cluster: false
tpu_use_sudo: false
use_cpu: false
"""

with open("configs/deepspeed_zero3.yaml", "w") as f:
    f.write(deepspeed_config)
print("✅ configs/deepspeed_zero3.yaml written")


In [ ]:
# Write the training recipe (hyperparameters)
os.makedirs("recipes", exist_ok=True)

recipe = """# Model
model_name_or_path: Qwen/Qwen2.5-3B-Instruct
model_revision: main
torch_dtype: bfloat16
attn_implementation: flash_attention_2
bf16: true
tf32: true
output_dir: runs/qwen-2.5-3b-r1-countdown

# Dataset
dataset_id_or_path: Jiayi-Pan/Countdown-Tasks-3to4

# Training
max_steps: 450
per_device_train_batch_size: 1
gradient_accumulation_steps: 8
gradient_checkpointing: true
gradient_checkpointing_kwargs:
  use_reentrant: false
learning_rate: 5.0e-7
lr_scheduler_type: cosine
warmup_ratio: 0.03

# GRPO
beta: 0.001
max_prompt_length: 256
max_completion_length: 1024
num_generations: 8
use_vllm: true
vllm_gpu_memory_utilization: 0.5

# Logging
logging_strategy: steps
logging_steps: 2
report_to:
  - tensorboard
save_strategy: steps
save_steps: 25
seed: 42
push_to_hub: false
"""

with open("recipes/grpo-countdown.yaml", "w") as f:
    f.write(recipe)
print("✅ recipes/grpo-countdown.yaml written")


In [ ]:
# Write the full training script
os.makedirs("scripts", exist_ok=True)

training_script = '''import logging
import os
from dataclasses import dataclass
from datetime import datetime
import random
import re
import torch
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
from transformers.trainer_utils import get_last_checkpoint
from transformers import AutoTokenizer
from datasets import load_dataset
from trl import GRPOConfig, GRPOTrainer, get_peft_config, ModelConfig, TrlParser


@dataclass
class ScriptArguments:
    dataset_id_or_path: str = "Jiayi-Pan/Countdown-Tasks-3to4"
    dataset_splits: str = "train"
    tokenizer_name_or_path: str = None


logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


def format_reward_func(completions, target, **kwargs):
    rewards = []
    for completion, gt in zip(completions, target):
        try:
            completion = "<think>" + completion
            if random.random() < 0.1:
                os.makedirs("completion_samples", exist_ok=True)
                with open("completion_samples/completion_samples.txt", "a") as f:
                    f.write(f"\\n\\n==============\\n" + completion)
            regex = r"^<think>([^<]*(?:<(?!/?think>)[^<]*)*)<\\/think>\\n<answer>([\\s\\S]*?)<\\/answer>$"
            match = re.search(regex, completion, re.DOTALL)
            if match is None or len(match.groups()) != 2:
                rewards.append(0.0)
            else:
                rewards.append(1.0)
        except Exception:
            rewards.append(0.0)
    return rewards


def equation_reward_func(completions, target, nums, **kwargs):
    rewards = []
    for completion, gt, numbers in zip(completions, target, nums):
        try:
            completion = "<think>" + completion
            match = re.search(r"<answer>(.*?)<\\/answer>", completion)
            if match is None:
                rewards.append(0.0)
                continue
            equation = match.group(1).strip()
            used_numbers = [int(n) for n in re.findall(r"\\d+", equation)]
            if sorted(used_numbers) != sorted(numbers):
                rewards.append(0.0)
                continue
            if not re.match(r"^[\\d+\\-*/().\\s]+$", equation):
                rewards.append(0.0)
                continue
            result = eval(equation, {"__builtins__": None}, {})
            if abs(float(result) - float(gt)) < 1e-5:
                rewards.append(1.0)
                if random.random() < 0.10:
                    os.makedirs("completion_samples", exist_ok=True)
                    with open("completion_samples/success_samples.txt", "a") as f:
                        f.write(f"\\n\\n==============\\n" + completion)
            else:
                rewards.append(0.0)
        except Exception:
            rewards.append(0.0)
    return rewards


def grpo_function(model_args, script_args, training_args):
    tokenizer = AutoTokenizer.from_pretrained(
        script_args.tokenizer_name_or_path or model_args.model_name_or_path,
        revision=model_args.model_revision,
        trust_remote_code=model_args.trust_remote_code,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    dataset = load_dataset(script_args.dataset_id_or_path, split=script_args.dataset_splits)
    dataset = dataset.shuffle(seed=42).select(range(50000))

    def generate_r1_prompt(numbers, target):
        msgs = [
            {"role": "system", "content": "You are a helpful assistant. You first thinks about the reasoning process in the mind and then provides the user with the answer."},
            {"role": "user", "content": f"Using the numbers {numbers}, create an equation that equals {target}. You can use basic arithmetic operations (+, -, *, /) one or multiple times but each number can only be used once. Show your work in <think> </think> tags. And return the final equation in <answer> </answer> tags, for example <answer> (1 + 2) / 3 </answer>. Think step by step inside <think> tags."},
            {"role": "assistant", "content": "Let me solve this step by step.\\n<think>"},
        ]
        return {"prompt": tokenizer.apply_chat_template(msgs, tokenize=False, continue_final_message=True), "target": target, "nums": numbers}

    dataset = dataset.map(lambda x: generate_r1_prompt(x["nums"], x["target"]))
    split = dataset.train_test_split(test_size=0.1)

    trainer = GRPOTrainer(
        model=model_args.model_name_or_path,
        reward_funcs=[format_reward_func, equation_reward_func],
        args=training_args,
        train_dataset=split["train"],
        eval_dataset=split["test"],
        peft_config=get_peft_config(model_args),
    )

    last_checkpoint = get_last_checkpoint(training_args.output_dir) if os.path.isdir(training_args.output_dir) else None
    logger.info(f"*** Starting training {datetime.now()} ***")
    train_result = trainer.train(resume_from_checkpoint=last_checkpoint)
    trainer.log_metrics("train", train_result.metrics)
    trainer.save_metrics("train", train_result.metrics)
    trainer.save_state()
    trainer.model.config.use_cache = True
    trainer.save_model(training_args.output_dir)
    tokenizer.save_pretrained(training_args.output_dir)
    training_args.distributed_state.wait_for_everyone()
    logger.info("*** Training complete! ***")


def main():
    parser = TrlParser((ModelConfig, ScriptArguments, GRPOConfig))
    model_args, script_args, training_args = parser.parse_args_and_config()
    grpo_function(model_args, script_args, training_args)


if __name__ == "__main__":
    main()
'''

with open("scripts/run_r1_grpo.py", "w") as f:
    f.write(training_script)
print("✅ scripts/run_r1_grpo.py written")
print(f"   Size: {len(training_script):,} characters")


### Launch Command

SSH into your multi-GPU machine and run:

```bash
# For 8 GPUs (7 train + 1 vLLM)
accelerate launch \\
    --num_processes 7 \\
    --config_file configs/deepspeed_zero3.yaml \\
    scripts/run_r1_grpo.py \\
    --config recipes/grpo-countdown.yaml
```

| GPUs | `num_processes` | Training Time (450 steps) |
|------|----------------|---------------------------|
| 4x H100 | 3 | ~6 hours |
| 8x A100 | 7 | ~3 hours |
| 4x A100 | 3 | ~8 hours |


<a name="9"></a>
## 9. 🔍 Inspecting Results

After training, the model saves completion samples to the `completion_samples/` directory. Let's look at what the model learned.


In [ ]:
# Show sample completions (from training logs)
# After real training, these files will exist:
import os

sample_files = [
    "completion_samples/completion_samples.txt",
    "completion_samples/success_samples.txt",
]

for fpath in sample_files:
    if os.path.exists(fpath):
        with open(fpath) as f:
            content = f.read()
        samples = content.split("==============")
        print(f"\n{'=' * 60}")
        print(f"File: {fpath} ({len(samples)-1} samples)")
        print(f"{'=' * 60}")
        # Show last 3 samples (most recent = most trained)
        for s in samples[-4:-1]:
            if s.strip():
                print(s.strip()[:500])
                print("-" * 40)
    else:
        print(f"\n{fpath} not found yet (run training first)")


### What to look for in the outputs:

| Training Stage | What You'll See |
|---|---|
| **Steps 0-50** | Model learns to use `<think>` and `<answer>` tags |
| **Steps 50-100** | First correct equations appear (~25% accuracy) |
| **Steps 100-300** | Accuracy climbs to ~50%, reasoning gets longer |
| **Steps 300+** | Self-correction: "Wait, that's wrong. Let me try..." 💡 |


<a name="10"></a>
## 10. 📊 Visualizing Training Progress

Let's create a visualization of expected training dynamics. After real training, you can load actual TensorBoard logs.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Simulated training curves (based on reported results from the blog)
np.random.seed(42)
steps = np.arange(0, 451, 2)

# Format reward: converges fast (sigmoid around step 30)
format_reward = 0.1 + 0.9 / (1 + np.exp(-(steps - 30) / 10))
format_reward += np.random.normal(0, 0.03, len(steps))
format_reward = np.clip(format_reward, 0, 1)

# Equation accuracy: slower sigmoid, starts around step 80
accuracy = 0.55 / (1 + np.exp(-(steps - 180) / 60))
accuracy += np.random.normal(0, 0.025, len(steps))
accuracy = np.clip(accuracy, 0, 1)

# Create the plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0a0a0a')

for ax in [ax1, ax2]:
    ax.set_facecolor('#121212')
    ax.tick_params(colors='#a3a3a3')
    ax.spines['bottom'].set_color('#262626')
    ax.spines['left'].set_color('#262626')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, alpha=0.1, color='white')

# Plot 1: Reward curves
ax1.plot(steps, format_reward, color='#00c8e6', linewidth=2, label='Format Reward', alpha=0.8)
ax1.plot(steps, accuracy, color='#66BB6A', linewidth=2, label='Equation Accuracy', alpha=0.8)
ax1.axvline(x=50, color='#a78bfa', linestyle='--', alpha=0.4, label='Format learned')
ax1.axvline(x=100, color='#FF9800', linestyle='--', alpha=0.4, label='25% accuracy')
ax1.set_xlabel('Training Steps', color='#a3a3a3')
ax1.set_ylabel('Average Reward', color='#a3a3a3')
ax1.set_title('Reward Curves', color='#f2f2f2', fontsize=14)
ax1.legend(facecolor='#1a1a1a', edgecolor='#262626', labelcolor='#a3a3a3')

# Plot 2: Combined reward over time
total_reward = format_reward + accuracy
ax2.fill_between(steps, 0, format_reward, color='#00c8e6', alpha=0.3, label='Format')
ax2.fill_between(steps, format_reward, total_reward, color='#66BB6A', alpha=0.3, label='Equation')
ax2.plot(steps, total_reward, color='#00d46a', linewidth=2, label='Total Reward')
ax2.set_xlabel('Training Steps', color='#a3a3a3')
ax2.set_ylabel('Stacked Reward', color='#a3a3a3')
ax2.set_title('Total Reward Progression', color='#f2f2f2', fontsize=14)
ax2.legend(facecolor='#1a1a1a', edgecolor='#262626', labelcolor='#a3a3a3')

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight', facecolor='#0a0a0a')
plt.show()
print("\nSaved to training_curves.png")


In [ ]:
# Milestone summary visualization
fig, ax = plt.subplots(figsize=(14, 3))
fig.patch.set_facecolor('#0a0a0a')
ax.set_facecolor('#0a0a0a')
ax.set_xlim(0, 450)
ax.set_ylim(-0.5, 1.5)
ax.axis('off')

# Timeline bar
from matplotlib.patches import FancyBboxPatch
bar = FancyBboxPatch((0, 0.4), 450, 0.2, boxstyle="round,pad=0.02",
                      facecolor='#262626', edgecolor='none')
ax.add_patch(bar)

# Gradient fill
for i in range(450):
    color_val = i / 450
    ax.axvline(x=i, ymin=0.3, ymax=0.5, color=plt.cm.viridis(color_val), alpha=0.6, linewidth=1)

# Milestones
milestones = [
    (50, 'Format\nLearned', '#00c8e6'),
    (100, '25%\nAccuracy', '#FF9800'),
    (200, '40%\nAccuracy', '#66BB6A'),
    (350, 'Reasoning\nEmerges', '#a78bfa'),
    (450, 'Training\nComplete', '#00d46a'),
]

for step, label, color in milestones:
    ax.plot(step, 0.5, 'o', color=color, markersize=12, zorder=5)
    ax.annotate(label, (step, 0.8), ha='center', va='bottom',
                fontsize=9, color=color, fontweight='bold')
    ax.annotate(f'Step {step}', (step, 0.15), ha='center', va='top',
                fontsize=8, color='#666')

ax.set_title('Training Milestones', color='#f2f2f2', fontsize=14, pad=20)
plt.tight_layout()
plt.show()


## 🧪 Exercises

### Exercise 1: Custom Reward Function
Add a third reward function that gives bonus reward for **shorter** reasoning chains (encourage conciseness).


In [ ]:
def length_reward_func(completions, target, **kwargs):
    """
    TODO: Implement a reward that penalizes overly long completions.
    - If completion < 200 chars: reward 1.0
    - If 200-500 chars: reward 0.5
    - If > 500 chars: reward 0.0
    """
    rewards = []
    for completion in completions:
        # TODO: your code here
        pass
    return rewards


### Exercise 2: Different Hyperparameters
Try changing `beta` (KL penalty) and observe the effect. What happens with `beta=0.1` vs `beta=0.0001`?


In [ ]:
# TODO: Create a new GRPOConfig with different beta values
# Compare: beta=0.1 (more conservative) vs beta=0.0001 (more exploration)
# What happens to format learning speed? Equation accuracy?

# config_conservative = GRPOConfig(beta=0.1, ...)
# config_exploratory = GRPOConfig(beta=0.0001, ...)


### Exercise 3: Harder Task
Modify the prompt to use 5-6 numbers instead of 3-4. Does the model still learn? How does training time change?


### Solutions


In [ ]:
# Solution to Exercise 1
def length_reward_func(completions, target, **kwargs):
    """Reward shorter, more concise reasoning chains."""
    rewards = []
    for completion in completions:
        length = len(completion)
        if length < 200:
            rewards.append(1.0)
        elif length < 500:
            rewards.append(0.5)
        else:
            rewards.append(0.0)
    return rewards

# Test it
test = ["short answer", "a" * 300, "a" * 600]
print(length_reward_func(test, ["65"] * 3))
# Expected: [1.0, 0.5, 0.0]


<a name="11"></a>
## 11. 🔑 Key Takeaways

- 💡 **RL can teach reasoning**: Models learn step-by-step thinking from reward signals alone — no chain-of-thought data needed
- 🎯 **GRPO > PPO for LLMs**: No critic model = half the memory. Group-relative advantages are simpler and equally effective
- ✅ **Rule-based rewards work**: Two binary checks (format + math correctness) are sufficient to drive emergent reasoning
- 🚀 **DeepSpeed + vLLM scale**: ZeRO-3 enables training 3B models on limited GPUs; vLLM makes GRPO rollouts practical
- ⚠️ **RL hyperparameters are sensitive**: Small changes in `beta` and `lr` dramatically affect training stability
- 📈 **Format before accuracy**: The model reliably learns XML tag structure (~50 steps) before mathematical correctness (~100+ steps)


<a name="12"></a>
## 12. 📚 Further Reading

- 📄 [DeepSeek R1 Paper](https://arxiv.org/abs/2501.12948) — The original paper describing the aha moment
- 📝 [Phil Schmid's Mini-R1 Blog Post](https://huggingface.co/blog/open-r1/mini-r1-contdown-game) — The tutorial this notebook is based on
- 📦 [TRL Documentation](https://huggingface.co/docs/trl/main/en/grpo_trainer) — GRPOTrainer API reference
- 🚀 [DeepSpeed ZeRO Tutorial](https://www.deepspeed.ai/tutorials/zero/) — Understanding ZeRO stages
- 🎮 [Countdown Game Dataset](https://huggingface.co/datasets/Jiayi-Pan/Countdown-Tasks-3to4) — The training dataset
- 🎥 [vLLM Documentation](https://docs.vllm.ai/) — High-throughput LLM serving

---

*Created by VizuaraAI — GPU Engineer's Bootcamp*
